In [25]:
"""
LongMemEval-S-Cleaned: Exploratory Data Analysis
==================================================
Analyzes token distributions, answer session patterns, and cross-checks
query answer_session_ids against docs across all 500 tenants.
"""

from datasets import load_dataset
from collections import Counter, defaultdict
import tiktoken

hf_path = "weaviate/longmemeval-s-cleaned"

print("Loading dataset...")
raw_docs = load_dataset(hf_path, "docs")["train"]
raw_queries = load_dataset(hf_path, "queries")["train"]

encoding = tiktoken.get_encoding("cl100k_base")

# --- Utility: Safe encoding with workaround for tiktoken special tokens error ---

def safe_encode(text, encoding):
    # Disable all special token checks (as recommended by the error message)
    # This allows e.g. <|endoftext|> to encode as a special token
    try:
        # tiktoken.encode API: disallowed_special defaults to encoding.special_tokens_set
        # We pass disallowed_special=() to prevent ValueError
        return encoding.encode(text, disallowed_special=())
    except Exception as e:
        print(f"Warning: failed to encode text: {e}")
        return []

# ── Pre-index by tenant (avoids O(n*m) filtering) ──
docs_by_tenant = defaultdict(list)
for item in raw_docs:
    docs_by_tenant[str(item["tenant_id"])].append(dict(item))

queries_by_tenant = defaultdict(list)
for item in raw_queries:
    queries_by_tenant[str(item["tenant_id"])].append(dict(item))

all_tenant_ids = sorted(docs_by_tenant.keys())
print(f"Found {len(all_tenant_ids)} tenants.\n")

# ── Build per-tenant stats ──
tenant_stats = []
all_doc_tokens = []  # flat list for global histogram

for tid in all_tenant_ids:
    docs = docs_by_tenant[tid]
    queries = queries_by_tenant.get(tid, [])

    token_counts = []
    answer_sessions = []

    for doc in docs:
        n_tokens = len(safe_encode(doc["session_text"], encoding))
        token_counts.append(n_tokens)
        all_doc_tokens.append(n_tokens)
        sid = doc["session_id"]
        if sid.startswith("answer_"):
            answer_sessions.append({"session_id": sid, "tokens": n_tokens})

    query_answer_ids = set()
    for q in queries:
        for aid in q["answer_session_ids"]:
            query_answer_ids.add(aid)

    tenant_stats.append({
        "tenant_id": tid,
        "num_docs": len(docs),
        "num_queries": len(queries),
        "num_answer_sessions": len(answer_sessions),
        "answer_sessions": answer_sessions,
        "query_answer_ids": sorted(query_answer_ids),
        "min_tokens": min(token_counts) if token_counts else 0,
        "max_tokens": max(token_counts) if token_counts else 0,
        "avg_tokens": sum(token_counts) / len(token_counts) if token_counts else 0,
        "total_tokens": sum(token_counts),
    })

# ═══════════════════════════════════════════════════════════════
# GLOBAL SUMMARY
# ═══════════════════════════════════════════════════════════════
total_docs = sum(t["num_docs"] for t in tenant_stats)
total_queries = sum(t["num_queries"] for t in tenant_stats)
total_tokens_all = sum(t["total_tokens"] for t in tenant_stats)

print("=" * 70)
print("LONGMEMEVAL-S-CLEANED: DATASET OVERVIEW")
print("=" * 70)
print(f"  Tenants:          {len(tenant_stats)}")
print(f"  Total docs:       {total_docs:,}")
print(f"  Total queries:    {total_queries:,}")
print(f"  Total tokens:     {total_tokens_all:,}")
print()

# ═══════════════════════════════════════════════════════════════
# TOKEN DISTRIBUTION ACROSS TENANTS
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("TOKEN DISTRIBUTION ACROSS TENANTS")
print("=" * 70)

all_totals = [t["total_tokens"] for t in tenant_stats]
all_avgs = [t["avg_tokens"] for t in tenant_stats]
all_mins = [t["min_tokens"] for t in tenant_stats]
all_maxs = [t["max_tokens"] for t in tenant_stats]
all_doc_counts = [t["num_docs"] for t in tenant_stats]

print(f"  {'Metric':<32} {'Min':>8} {'Avg':>8} {'Max':>8}")
print(f"  {'-'*56}")
print(f"  {'Docs per tenant':<32} {min(all_doc_counts):>8} {sum(all_doc_counts)/len(all_doc_counts):>8.0f} {max(all_doc_counts):>8}")
print(f"  {'Total tokens per tenant':<32} {min(all_totals):>8,} {sum(all_totals)/len(all_totals):>8,.0f} {max(all_totals):>8,}")
print(f"  {'Avg tokens per doc (by tenant)':<32} {min(all_avgs):>8.0f} {sum(all_avgs)/len(all_avgs):>8.0f} {max(all_avgs):>8.0f}")
print(f"  {'Min doc tokens (by tenant)':<32} {min(all_mins):>8} {sum(all_mins)/len(all_mins):>8.0f} {max(all_mins):>8}")
print(f"  {'Max doc tokens (by tenant)':<32} {min(all_maxs):>8} {sum(all_maxs)/len(all_maxs):>8.0f} {max(all_maxs):>8}")
print()

# ═══════════════════════════════════════════════════════════════
# QUERIES PER TENANT
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("QUERIES PER TENANT DISTRIBUTION")
print("=" * 70)
query_counts = [t["num_queries"] for t in tenant_stats]
qc_dist = Counter(query_counts)
for k in sorted(qc_dist.keys()):
    bar = "█" * qc_dist[k]
    print(f"  {k} queries: {qc_dist[k]:>4} tenants  {bar}")
print()

# ═══════════════════════════════════════════════════════════════
# ANSWER SESSION ANALYSIS
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("ANSWER SESSION ANALYSIS")
print("=" * 70)

answer_count_dist = Counter(t["num_answer_sessions"] for t in tenant_stats)
print("  answer_ sessions per tenant:")
for k in sorted(answer_count_dist.keys()):
    bar = "█" * min(answer_count_dist[k], 60)
    print(f"    {k} answer session(s): {answer_count_dist[k]:>4} tenants  {bar}")
print()

# Detail on multi-answer tenants
multi_answer = [t for t in tenant_stats if t["num_answer_sessions"] > 1]
if multi_answer:
    print(f"  TENANTS WITH MULTIPLE answer_ SESSIONS ({len(multi_answer)} found):")
    print(f"  {'-'*66}")
    for t in multi_answer:
        print(f"    Tenant {t['tenant_id']}  |  {t['num_docs']} docs, {t['num_queries']} queries")
        for a in t["answer_sessions"]:
            print(f"      → {a['session_id']}  ({a['tokens']} tokens)")
        print(f"      Query refs: {t['query_answer_ids']}")
        print()
else:
    print("  No tenants have multiple answer_ sessions.")
    print()

# ═══════════════════════════════════════════════════════════════
# ANSWER ID CROSS-CHECK
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("ANSWER ID CROSS-CHECK")
print("=" * 70)
print("  Checking: do all query answer_session_ids have a matching answer_ doc?")
print()

mismatches = 0
for t in tenant_stats:
    doc_answer_ids = set(a["session_id"] for a in t["answer_sessions"])
    query_answer_ids = set(t["query_answer_ids"])
    missing_in_docs = query_answer_ids - doc_answer_ids
    if missing_in_docs:
        print(f"  ✗ Tenant {t['tenant_id']}: query refs not found in docs: {missing_in_docs}")
        mismatches += 1

if mismatches == 0:
    print("  ✓ All query answer_session_ids have matching answer_ docs.")
print()

# Also check: answer_ docs not referenced by any query
unreferenced = 0
for t in tenant_stats:
    doc_answer_ids = set(a["session_id"] for a in t["answer_sessions"])
    query_answer_ids = set(t["query_answer_ids"])
    extra = doc_answer_ids - query_answer_ids
    if extra:
        unreferenced += 1
        if unreferenced <= 5:  # show first few
            print(f"  ℹ Tenant {t['tenant_id']}: answer_ docs not referenced by queries: {extra}")

if unreferenced > 5:
    print(f"  ... and {unreferenced - 5} more tenants with unreferenced answer_ docs")
if unreferenced == 0:
    print("  ✓ All answer_ docs are referenced by at least one query.")
print()

# ═══════════════════════════════════════════════════════════════
# PER-DOC TOKEN SIZE HISTOGRAM
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("PER-DOC TOKEN SIZE HISTOGRAM (all docs, all tenants)")
print("=" * 70)

buckets = [
    (0, 100), (100, 500), (500, 1000), (1000, 2000),
    (2000, 3000), (3000, 4000), (4000, 5000), (5000, 10000)
]
print(f"  {'Range':<20} {'Count':>8} {'%':>8}")
print(f"  {'-'*40}")
for lo, hi in buckets:
    count = sum(1 for t in all_doc_tokens if lo <= t < hi)
    pct = count / len(all_doc_tokens) * 100
    bar = "█" * int(pct / 2)
    print(f"  {lo:>5}-{hi:<5} tokens  {count:>8} {pct:>7.1f}%  {bar}")

print(f"\n  Total docs: {len(all_doc_tokens):,}")
print(f"  Global min: {min(all_doc_tokens):,}  max: {max(all_doc_tokens):,}  avg: {sum(all_doc_tokens)/len(all_doc_tokens):,.0f}")
print()

# ═══════════════════════════════════════════════════════════════
# ANSWER vs NON-ANSWER TOKEN COMPARISON
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("ANSWER vs NON-ANSWER DOC TOKEN COMPARISON")
print("=" * 70)

answer_tokens = []
non_answer_tokens = []
for tid in all_tenant_ids:
    for doc in docs_by_tenant[tid]:
        n = len(safe_encode(doc["session_text"], encoding))
        if doc["session_id"].startswith("answer_"):
            answer_tokens.append(n)
        else:
            non_answer_tokens.append(n)

print(f"  {'Type':<20} {'Count':>8} {'Min':>8} {'Avg':>8} {'Max':>8}")
print(f"  {'-'*54}")
if answer_tokens:
    print(f"  {'answer_ docs':<20} {len(answer_tokens):>8} {min(answer_tokens):>8} {sum(answer_tokens)//len(answer_tokens):>8} {max(answer_tokens):>8}")
if non_answer_tokens:
    print(f"  {'non-answer docs':<20} {len(non_answer_tokens):>8} {min(non_answer_tokens):>8} {sum(non_answer_tokens)//len(non_answer_tokens):>8} {max(non_answer_tokens):>8}")
print()

Loading dataset...
Found 500 tenants.

LONGMEMEVAL-S-CLEANED: DATASET OVERVIEW
  Tenants:          500
  Total docs:       23,867
  Total queries:    500
  Total tokens:     52,033,286

TOKEN DISTRIBUTION ACROSS TENANTS
  Metric                                Min      Avg      Max
  --------------------------------------------------------
  Docs per tenant                        38       48       62
  Total tokens per tenant            97,466  104,067  106,131
  Avg tokens per doc (by tenant)       1666     2192     2749
  Min doc tokens (by tenant)              6      194      816
  Max doc tokens (by tenant)           3356     4288    17263

QUERIES PER TENANT DISTRIBUTION
  1 queries:  500 tenants  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

In [26]:
from datasets import load_dataset
import tiktoken

tenant_id = None

hf_path = "weaviate/longmemeval-s-cleaned"

raw_docs = load_dataset(hf_path, "docs")["train"]
raw_queries = load_dataset(hf_path, "queries")["train"]

all_tenant_ids = sorted(set(str(item["tenant_id"]) for item in raw_docs))
print(f"Found {len(all_tenant_ids)} tenants.")

if tenant_id is None:
    tenant_id = all_tenant_ids[0]
    print(f"No tenant_id specified, using first tenant: {tenant_id}")
elif tenant_id not in all_tenant_ids:
    raise ValueError(f"Tenant ID '{tenant_id}' not found. Try one of: {all_tenant_ids[:5]}")

tenant_docs = [dict(item) for item in raw_docs if str(item["tenant_id"]) == tenant_id]
tenant_queries = [dict(item) for item in raw_queries if str(item["tenant_id"]) == tenant_id]

print(f"Tenant '{tenant_id}': {len(tenant_docs)} docs, {len(tenant_queries)} queries")

# Choose a token encoding; GPT-3.5 default used here, adjust as needed
encoding = tiktoken.get_encoding("cl100k_base")

if tenant_docs:
    print("\nDocs for this tenant (number of tokens):")
    for i, doc in enumerate(tenant_docs, 1):
        doc_text = doc["session_text"]
        session_id = doc["session_id"]
        num_tokens = len(encoding.encode(doc_text))
        print(f"{i}. {num_tokens} tokens for session: {session_id}.")

if tenant_queries:
    q = tenant_queries[0]
    print(f"\nSample Question: {q['question']}\nAnswer session IDs: {q['answer_session_ids']}")

    print("\nQueries for this tenant:")
    for i, q in enumerate(tenant_queries, 1):
        print(f"{i}. {q['question']}\n   Answer session IDs: {q['answer_session_ids']}")

Found 500 tenants.
No tenant_id specified, using first tenant: 001be529
Tenant '001be529': 51 docs, 1 queries

Docs for this tenant (number of tokens):
1. 3383 tokens for session: 3722ea11_2.
2. 2411 tokens for session: sharegpt_6cz1Sq6_328.
3. 2902 tokens for session: 3c11dc79.
4. 2278 tokens for session: c8854b28.
5. 905 tokens for session: sharegpt_YSEjnMT_7.
6. 3070 tokens for session: 0eb28b73_2.
7. 3509 tokens for session: f067b0ee.
8. 337 tokens for session: sharegpt_u1DIJRf_0.
9. 1801 tokens for session: sharegpt_ETJwE0s_0.
10. 141 tokens for session: sharegpt_HbIK81o_0.
11. 992 tokens for session: ultrachat_499876.
12. 1601 tokens for session: ultrachat_351532.
13. 3361 tokens for session: 0f71db75_1.
14. 3816 tokens for session: 28e0ea23_2.
15. 3431 tokens for session: ed9dad6c_1.
16. 2259 tokens for session: afabb63b.
17. 1259 tokens for session: ultrachat_471978.
18. 4356 tokens for session: 93aff0d7_1.
19. 1433 tokens for session: ultrachat_241736.
20. 2029 tokens for sess

In [28]:
answer_doc = next((doc for doc in tenant_docs if doc.get("session_id", "").startswith("answer_")), None)

print(len(encoding.encode(answer_doc["session_text"])))

3165


In [36]:
print(answer_doc["session_text"])

user: I'm trying to find a permanent place to live. Can you help me with that? I'm on a limited budget and not sure where to start looking.
assistant: I'd be happy to help you find a permanent place to live on a limited budget. Let's break down the process into manageable steps.

**Step 1: Assess Your Budget**

To start, can you please share some details about your budget? This will help me provide more tailored advice:

1. What is your maximum monthly rent budget?
2. Do you have any savings for a security deposit, first month's rent, and moving costs?
3. Are you eligible for any government assistance programs, such as Section 8 housing?

**Step 2: Identify Your Priorities**

What are your must-haves in a living situation? Consider the following:

1. Location: Do you need to be close to work, public transportation, schools, or family/friends?
2. Type of accommodation: Apartment, house, room rental, or shared living situation?
3. Amenities: Do you need access to a yard, gym, pool, or in

In [30]:
from engram import EngramClient
import os

client = EngramClient(
    api_key=os.environ["ENGRAM_API_KEY"],
    base_url="https://dev-engram.labs.weaviate.io"
)

In [31]:
run = client.memories.add(
    answer_doc["session_text"],
    user_id="connors-first-test",
    group="default",
)

print(run.run_id)
print(run.status)

019d25ce-8d70-7a8d-b6f5-5f34a3a6c29c
running


In [33]:
status = client.runs.wait(run.run_id)

print(status.run_id)
print(status.status)
print(status.committed_operations)

# If completed, parse memory ids into a list
created_memories = []
if getattr(status, "status", None) == "completed" and hasattr(status, "committed_operations"):
    # status.committed_operations.created is likely a list of CommittedOperation objects
    created = getattr(status.committed_operations, "created", [])
    created_memories = [op.memory_id for op in created if hasattr(op, "memory_id")]
    print("created_memories:", created_memories)

019d25ce-8d70-7a8d-b6f5-5f34a3a6c29c
completed
CommittedOperations(created=[CommittedOperation(memory_id='6b564597-dfd9-4c86-9204-127d565215f7', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='35eea2db-725e-4743-99fd-a05e56a81a86', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='a498d030-3c85-47ee-92eb-7f36791d8bf0', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='b4a4a1fd-fb12-4cd4-9397-65b0e2f9f948', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='5a501b2f-3ac4-48da-82df-18a3d32fdd48', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='1e45f0c3-3c92-4fd3-b45f-0be9da232edd', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='e09aca66-196e-4daa-bf0f-6558c1d242a7', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='bafeb25a-b63a-4817-85bf-0a0a15b862a5', committed_at='2026-03-25T16:23:19Z'), CommittedOperation(memory_id='ec07f354-642b-4f77-a96f-bde0a29759f4',

In [35]:
for memory_id in created_memories:
    memory = client.memories.get(memory_id, user_id="connors-first-test", group="default")
    print(memory.content)

The user is not eligible for any government assistance programs for housing.
The user is open to any type of accommodation as long as it is safe and clean.
The user has been using Zillow and HotPads to search for apartments.
The user’s asylum application was approved after taking over a year.
The user has a maximum monthly rent budget of $800.
The user has some savings for a security deposit and first month's rent.
The user needs housing close to public transportation.
The user needs housing close to their volunteer job.
The user is volunteering at a nonprofit organization that helps refugees.
The user is currently focusing their housing search on "Neighborhood 1" but also wants to expand the search to surrounding neighborhoods.
The user is trying to find a permanent place to live.


## Engram update with context

In [38]:
for tenant_doc in tenant_docs:
    run_id = client.memories.add(
        tenant_doc["session_text"],
        user_id="with-context-test",
        group="default",
    )
    if tenant_doc["session_id"].startswith("answer_"):
        answer_session_engram_run_id = run_id
        print(f"Tracking {answer_session_engram_run_id.run_id}")

Tracking 019d25e5-b317-7ccc-b9f9-fcf3eb5c0736


In [47]:
status = client.runs.wait(answer_session_engram_run_id.run_id)

print(status.run_id)
print(status.status)
print(status.committed_operations)

# If completed, parse memory ids into a list
created_memories = []
if getattr(status, "status", None) == "completed" and hasattr(status, "committed_operations"):
    # status.committed_operations.created is likely a list of CommittedOperation objects
    created = getattr(status.committed_operations, "created", [])
    created_memories = [op.memory_id for op in created if hasattr(op, "memory_id")]
    print("created_memories:", created_memories)

019d25e5-b317-7ccc-b9f9-fcf3eb5c0736
completed
CommittedOperations(created=[CommittedOperation(memory_id='44547d77-d6db-443c-b334-20a236f006be', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='482ebcc5-7489-4cda-81b7-d1b30e0c7162', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='728c3a62-9f60-44c1-a448-c02603ebb598', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='510f1ab9-b602-4a60-91fd-224441081c0f', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='58baf549-edf3-462d-95ac-8d8757d567b8', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='942599dc-3844-482d-aa8d-95a6f7f93f74', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='dc7b0552-50cf-4f73-b80d-addd0e85540d', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='35b065ea-0d6a-4c9a-b6f9-a844d3672b27', committed_at='2026-03-25T17:04:23Z'), CommittedOperation(memory_id='125cd811-2dc9-4730-ba6b-2c91409b5009',

In [48]:
for memory_id in created_memories:
    memory = client.memories.get(memory_id, user_id="with-context-test", group="default")
    print(memory.content)

User is trying to find a permanent place to live and is on a limited budget.
User's asylum application was approved after taking over a year.
User has savings for a security deposit and first month's rent.
User needs housing close to public transportation.
User's maximum monthly rent budget is $800.
User's volunteer work helps them give back, meet like-minded people, and build their resume and US work experience.
User needs housing close to their volunteer job.
User has been using Zillow to search for apartments.
User is not eligible for government assistance programs (e.g., Section 8).
User is open to any type of accommodation as long as it is safe and clean.
